# Configuration

> Configuration dataclasses for file scanning including ScanConfig, FilterConfig, and ExtensionMapping.

In [ ]:
#| default_exp core.config

In [ ]:
#| export
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Set

from cjm_file_discovery.core.models import FileType, FileInfo

## ExtensionMapping

Maps file extensions to FileType categories. Used to automatically categorize files based on their extension.

In [ ]:
#| export
@dataclass
class ExtensionMapping:
    """Maps file extensions to FileType categories."""
    audio: List[str] = field(default_factory=lambda: [
        'mp3', 'wav', 'flac', 'aac', 'm4a', 'ogg', 'wma', 'opus'
    ])
    video: List[str] = field(default_factory=lambda: [
        'mp4', 'mkv', 'avi', 'mov', 'wmv', 'webm', 'flv', 'm4v'
    ])
    image: List[str] = field(default_factory=lambda: [
        'jpg', 'jpeg', 'png', 'gif', 'bmp', 'webp', 'svg', 'ico', 'tiff', 'tif'
    ])
    document: List[str] = field(default_factory=lambda: [
        'pdf', 'doc', 'docx', 'txt', 'rtf', 'odt', 'md', 'rst'
    ])
    code: List[str] = field(default_factory=lambda: [
        'py', 'js', 'ts', 'html', 'css', 'java', 'cpp', 'c', 'h', 'go', 'rs',
        'rb', 'php', 'swift', 'kt', 'scala', 'sh', 'bash', 'zsh'
    ])
    data: List[str] = field(default_factory=lambda: [
        'json', 'xml', 'csv', 'yaml', 'yml', 'db', 'sqlite', 'sqlite3', 'sql', 'toml'
    ])
    archive: List[str] = field(default_factory=lambda: [
        'zip', 'tar', 'gz', 'bz2', '7z', 'rar', 'xz', 'tgz'
    ])

    def get_type(
        self,
        extension: str  # File extension (with or without dot)
    ) -> FileType:  # Corresponding FileType
        """Get FileType for an extension."""
        ext = extension.lower().lstrip('.')
        ext_map = self.build_extension_map()
        return ext_map.get(ext, FileType.OTHER)

    def build_extension_map(self) -> Dict[str, FileType]:  # Mapping of extension to FileType
        """Build reverse mapping from extension to FileType."""
        ext_map = {}
        for ext in self.audio:
            ext_map[ext.lower()] = FileType.AUDIO
        for ext in self.video:
            ext_map[ext.lower()] = FileType.VIDEO
        for ext in self.image:
            ext_map[ext.lower()] = FileType.IMAGE
        for ext in self.document:
            ext_map[ext.lower()] = FileType.DOCUMENT
        for ext in self.code:
            ext_map[ext.lower()] = FileType.CODE
        for ext in self.data:
            ext_map[ext.lower()] = FileType.DATA
        for ext in self.archive:
            ext_map[ext.lower()] = FileType.ARCHIVE
        return ext_map

    def get_all_extensions(self) -> Set[str]:  # Set of all known extensions
        """Get all configured extensions."""
        return set(
            self.audio + self.video + self.image + self.document +
            self.code + self.data + self.archive
        )

In [ ]:
# Test ExtensionMapping
mapping = ExtensionMapping()

assert mapping.get_type("mp3") == FileType.AUDIO
assert mapping.get_type(".MP3") == FileType.AUDIO  # Case insensitive, with dot
assert mapping.get_type("mp4") == FileType.VIDEO
assert mapping.get_type("png") == FileType.IMAGE
assert mapping.get_type("pdf") == FileType.DOCUMENT
assert mapping.get_type("py") == FileType.CODE
assert mapping.get_type("json") == FileType.DATA
assert mapping.get_type("zip") == FileType.ARCHIVE
assert mapping.get_type("unknown") == FileType.OTHER

ext_map = mapping.build_extension_map()
assert "mp3" in ext_map
assert ext_map["mp3"] == FileType.AUDIO

all_exts = mapping.get_all_extensions()
assert "mp3" in all_exts
assert "mp4" in all_exts
assert len(all_exts) > 50  # Should have many extensions

print("ExtensionMapping tests passed!")

ExtensionMapping tests passed!


## FilterConfig

Configuration for filtering files during discovery.

In [ ]:
#| export
@dataclass
class FilterConfig:
    """Configuration for filtering files during discovery."""
    extensions: Optional[List[str]] = None          # Include only these extensions (None = all)
    exclude_extensions: Optional[List[str]] = None  # Exclude these extensions
    file_types: Optional[List[FileType]] = None     # Include only these types (None = all)
    min_size: Optional[int] = None                  # Minimum file size (bytes)
    max_size: Optional[int] = None                  # Maximum file size (bytes)
    exclude_patterns: List[str] = field(default_factory=list)  # Glob patterns to exclude
    include_hidden: bool = False                    # Include hidden files/directories
    custom_filter: Optional[Callable[[FileInfo], bool]] = None  # Custom filter function

    def matches(
        self,
        file_info: FileInfo,  # File to check
        extension_mapping: Optional[ExtensionMapping] = None  # For type checking
    ) -> bool:  # True if file passes all filters
        """Check if a file matches all filter criteria."""
        # Extension filter
        if self.extensions is not None:
            if file_info.extension is None:
                return False
            if file_info.extension.lower() not in [e.lower().lstrip('.') for e in self.extensions]:
                return False

        # Exclude extensions
        if self.exclude_extensions is not None:
            if file_info.extension is not None:
                if file_info.extension.lower() in [e.lower().lstrip('.') for e in self.exclude_extensions]:
                    return False

        # File type filter
        if self.file_types is not None:
            if file_info.file_type not in self.file_types:
                return False

        # Size filters
        if self.min_size is not None and file_info.size is not None:
            if file_info.size < self.min_size:
                return False
        if self.max_size is not None and file_info.size is not None:
            if file_info.size > self.max_size:
                return False

        # Exclude patterns
        if self.exclude_patterns:
            from cjm_file_discovery.utils.formatting import matches_glob_patterns
            if matches_glob_patterns(file_info.path, self.exclude_patterns):
                return False

        # Hidden files
        if not self.include_hidden and file_info.name.startswith('.'):
            return False

        # Custom filter
        if self.custom_filter is not None:
            if not self.custom_filter(file_info):
                return False

        return True

In [ ]:
# Test FilterConfig
filter_cfg = FilterConfig(extensions=['py', 'js'])
py_file = FileInfo(name="test.py", path="/test.py", is_directory=False, extension="py")
txt_file = FileInfo(name="test.txt", path="/test.txt", is_directory=False, extension="txt")

assert filter_cfg.matches(py_file) == True
assert filter_cfg.matches(txt_file) == False

# Test file type filter
type_filter = FilterConfig(file_types=[FileType.AUDIO, FileType.VIDEO])
audio_file = FileInfo(name="song.mp3", path="/song.mp3", is_directory=False, file_type=FileType.AUDIO)
image_file = FileInfo(name="pic.png", path="/pic.png", is_directory=False, file_type=FileType.IMAGE)

assert type_filter.matches(audio_file) == True
assert type_filter.matches(image_file) == False

# Test size filter
size_filter = FilterConfig(min_size=1000, max_size=10000)
small_file = FileInfo(name="small.txt", path="/small.txt", is_directory=False, size=500)
medium_file = FileInfo(name="medium.txt", path="/medium.txt", is_directory=False, size=5000)
large_file = FileInfo(name="large.txt", path="/large.txt", is_directory=False, size=50000)

assert size_filter.matches(small_file) == False
assert size_filter.matches(medium_file) == True
assert size_filter.matches(large_file) == False

# Test hidden files
hidden_filter = FilterConfig(include_hidden=False)
hidden_file = FileInfo(name=".hidden", path="/.hidden", is_directory=False)
normal_file = FileInfo(name="normal.txt", path="/normal.txt", is_directory=False)

assert hidden_filter.matches(hidden_file) == False
assert hidden_filter.matches(normal_file) == True

# Test include hidden
hidden_filter_include = FilterConfig(include_hidden=True)
assert hidden_filter_include.matches(hidden_file) == True

print("FilterConfig tests passed!")

FilterConfig tests passed!


## ScanConfig

Main configuration for file scanning operations.

In [ ]:
#| export
@dataclass
class ScanConfig:
    """Main configuration for file scanning operations."""
    directories: List[str] = field(default_factory=list)  # Directories to scan
    recursive: bool = True                                 # Scan subdirectories
    max_depth: Optional[int] = None                        # Maximum recursion depth (None = unlimited)
    follow_symlinks: bool = False                          # Follow symbolic links
    filter_config: FilterConfig = field(default_factory=FilterConfig)
    extension_mapping: ExtensionMapping = field(default_factory=ExtensionMapping)

    # Caching
    cache_results: bool = True
    cache_duration_seconds: int = 300                      # 5 minutes default

    # Limits
    max_results: Optional[int] = None                      # Maximum files to return (None = unlimited)

    # Sorting
    sort_by: str = "name"                                  # name, size, modified, type
    sort_descending: bool = False

In [ ]:
# Test ScanConfig
config = ScanConfig(
    directories=["/home/user/documents", "/home/user/downloads"],
    recursive=True,
    max_depth=3,
    filter_config=FilterConfig(extensions=['pdf', 'doc']),
    cache_duration_seconds=600,
    sort_by="modified"
)

assert len(config.directories) == 2
assert config.recursive == True
assert config.max_depth == 3
assert config.filter_config.extensions == ['pdf', 'doc']
assert config.cache_duration_seconds == 600
assert config.sort_by == "modified"

# Test default values
default_config = ScanConfig()
assert default_config.directories == []
assert default_config.recursive == True
assert default_config.max_depth is None
assert default_config.cache_results == True
assert default_config.sort_by == "name"

print("ScanConfig tests passed!")

ScanConfig tests passed!


In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()